# Asteria / Cortex — Semantic-Aware Cross-Region Caching for LLM Agents

This notebook implements the core architecture from the paper *"Cortex: Achieving Low-Latency, Cost-Efficient Remote Data Access For LLM via Semantic-Aware Knowledge Caching"* (arXiv 2509.17360).

## What this notebook covers

| Section | What you build |
|---|---|
| 1 | Setup & dependencies |
| 2 | Semantic Element (SE) — the cache unit |
| 3 | Embedding model (real or simulated) |
| 4 | Sine — two-stage retrieval (ANN + Semantic Judger) |
| 5 | Cache architecture (LCFU eviction, TTL, prefetching) |
| 6 | Agent simulation & the full caching pipeline |
| 7 | Experiments: hit rate, latency, cost, accuracy |
| 8 | Ablations: ANN-only vs full Asteria vs exact-match |

## Two operating modes

- **Simulation mode** (default) — runs on any CPU. Uses `sentence-transformers` for embeddings and a rule-based judger. No GPU needed.
- **Real model mode** — swap in `Qwen3-Embedding-0.6B` and `Qwen3-Reranker-0.6B` via vLLM or HuggingFace. Toggle `USE_REAL_MODELS = True`.

## Section 1 — Setup

In [1]:
# Install dependencies — run once then restart kernel
!pip install sentence-transformers faiss-cpu numpy pandas matplotlib seaborn tqdm datasets transformers torch -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 51.3 MB/s eta 0:00:00


In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import faiss
import time
import random
import json
import hashlib
import heapq
from dataclasses import dataclass, field
from typing import Optional, Dict, List, Tuple, Any
from collections import defaultdict, deque
from copy import deepcopy
from tqdm import tqdm

# ── Configuration ─────────────────────────────────────────────────────────────
# Paper: Qwen3-Embedding-0.6B (1024-dim) + Qwen3-Reranker-0.6B
EMBEDDING_DIM        = 1024    # Qwen3-Embedding-0.6B output dimension
ANN_SIM_THRESH       = 0.75    # τ_sim  — coarse ANN threshold
LSM_CONF_THRESH      = 0.80    # τ_lsm  — semantic judger confidence threshold
STATICITY_VOLATILE   = 2.0     # SEs with model-predicted staticity ≤ this are NOT cached
CACHE_CAPACITY       = 50      # Max number of SEs in cache
MARKOV_THETA         = 0.30    # Prefetch confidence threshold (Algorithm 3)
REMOTE_LATENCY_MS    = 350     # Simulated cross-region API latency
REMOTE_COST_PER_CALL = 0.005   # $ per API call
SEED                 = 42
random.seed(SEED)
np.random.seed(SEED)

print("Configuration loaded.")
print(f"  Embedding model : Qwen/Qwen3-Embedding-0.6B  dim={EMBEDDING_DIM}")
print(f"  Judger model    : Qwen/Qwen3-Reranker-0.6B")
print(f"  Cache capacity  : {CACHE_CAPACITY} SEs")
print(f"  τ_sim           : {ANN_SIM_THRESH}")
print(f"  τ_lsm           : {LSM_CONF_THRESH}")
print(f"  Markov θ        : {MARKOV_THETA}")


Configuration loaded.
  Embedding model : Qwen/Qwen3-Embedding-0.6B  dim=1024
  Judger model    : Qwen/Qwen3-Reranker-0.6B
  Cache capacity  : 50 SEs
  τ_sim           : 0.75
  τ_lsm           : 0.8
  Markov θ        : 0.3


## Section 2 — Semantic Element (SE)

The **Semantic Element** is the fundamental caching unit. It wraps:
- The agent's query (semantic key)
- The retrieved answer (value)
- A dense embedding vector (for ANN search)
- Performance metadata: cost, latency, access frequency, staticity, size, TTL

In [3]:
import time

@dataclass
class SemanticElement:
    """
    Core caching unit — maps directly to Figure 5 in the paper.

    Semantic Key  : query string  (e.g. "Who painted the Mona Lisa?")
    Value         : retrieved answer string
    Embedding     : dense float32 vector from the embedding model
    Metadata      : cost, latency, frequency, staticity, size, ttl
    """
    query:      str
    answer:     str
    embedding:  np.ndarray          # shape (EMBEDDING_DIM,)

    # ── Performance metadata ─────────────────────────────────────────
    cost:       float = REMOTE_COST_PER_CALL  # $ cost of original API call
    latency_ms: float = REMOTE_LATENCY_MS     # ms of original API call
    staticity:  float = 5.0    # 1–10: how time-stable is this fact?
    frequency:  int   = 0      # times this SE was a confirmed cache hit
    size_tokens:int   = 0      # token length of answer (proxy for memory)
    created_at: float = field(default_factory=time.time)
    ttl_seconds:float = 3600.0 # time-to-live in seconds

    # ── Internal index tracking ───────────────────────────────────────
    faiss_id:   int   = -1     # position in FAISS index (set on insertion)

    def __post_init__(self):
        if self.size_tokens == 0:
            self.size_tokens = max(1, len(self.answer.split()))

    @property
    def is_expired(self) -> bool:
        return (time.time() - self.created_at) > self.ttl_seconds

    @property
    def lcfu_score(self) -> float:
        """
        LCFU value score from Algorithm 2 in the paper.

        score = log(freq+1) × log(cost×1000+1) × log(latency+1) × log(staticity+1)
                ─────────────────────────────────────────────────────────────────────
                                        size_tokens

        Higher score → more valuable → less likely to be evicted.
        Logarithms prevent low-cost items from being penalised and keep score positive.
        """
        ttl_remaining = self.ttl_seconds - (time.time() - self.created_at)
        if self.size_tokens == 0 or ttl_remaining <= 0:
            return 0.0
        score = (
            np.log1p(self.frequency) *
            np.log1p(self.cost * 1000) *
            np.log1p(self.latency_ms) *
            np.log1p(self.staticity)
        ) / self.size_tokens
        return score

    def summary(self) -> dict:
        return {
            "query":       self.query[:60],
            "answer":      self.answer[:80],
            "staticity":   self.staticity,
            "frequency":   self.frequency,
            "cost_$":      self.cost,
            "latency_ms":  self.latency_ms,
            "lcfu_score":  round(self.lcfu_score, 4),
            "expired":     self.is_expired,
        }


# ── Quick demo ───────────────────────────────────────────────────────────────
dummy_embedding = np.random.randn(EMBEDDING_DIM).astype(np.float32)
se_demo = SemanticElement(
    query="Who painted the Mona Lisa?",
    answer="Leonardo da Vinci painted the Mona Lisa during the Renaissance.",
    embedding=dummy_embedding,
    staticity=9.5,   # historical fact — very stable
    cost=0.005,
    latency_ms=320,
)

print("Semantic Element demo:")
for k, v in se_demo.summary().items():
    print(f"  {k:<15}: {v}")

Semantic Element demo:
  query          : Who painted the Mona Lisa?
  answer         : Leonardo da Vinci painted the Mona Lisa during the Renaissance.
  staticity      : 9.5
  frequency      : 0
  cost_$         : 0.005
  latency_ms     : 320
  lcfu_score     : 0.0
  expired        : False


## Section 3 — Embedding Model

Converts query strings → dense vectors used for ANN similarity search.

- **Simulation mode**: `all-MiniLM-L6-v2` (22M params, CPU-friendly, 384-dim)
- **Real model mode**: `Qwen/Qwen3-Embedding-0.6B` (600M params, 1024-dim, paper's choice)

In [4]:
from sentence_transformers import SentenceTransformer

class EmbeddingModel:
    """
    Wraps Qwen3-Embedding-0.6B via the SentenceTransformer interface.

    SentenceTransformer automatically applies last-token pooling (correct for
    decoder-based Qwen3 — NOT CLS pooling which is wrong for this architecture).
    Always returns L2-normalised float32 vectors of dim=1024.

    Paper reference: Section 4.1 — Qwen3-Embedding-0.6B encodes queries into
    dense vectors for ANN search in the Sine index.
    """

    def __init__(self):
        model_name = "Qwen/Qwen3-Embedding-0.6B"
        # SentenceTransformer reads the model card config and applies the
        # correct pooling strategy automatically — last-token for Qwen3.
        self.model = SentenceTransformer(model_name)
        self.dim   = 1024
        print(f"Loaded embedding model: {model_name}  dim={self.dim}")

    def encode(self, texts: List[str]) -> np.ndarray:
        """Returns shape (N, 1024) float32 L2-normalised embeddings."""
        vecs = self.model.encode(
            texts,
            show_progress_bar=False,
            normalize_embeddings=True,   # L2 normalise so cosine = inner product
            batch_size=16,
        )
        return vecs.astype(np.float32)

    def encode_one(self, text: str) -> np.ndarray:
        return self.encode([text])[0]


embedding_model = EmbeddingModel()

# Sanity check
v1 = embedding_model.encode_one("Who painted the Mona Lisa?")
v2 = embedding_model.encode_one("What artist created the Mona Lisa?")
v3 = embedding_model.encode_one("What is the capital of France?")
print(f"\nSimilarity (same topic):  {np.dot(v1, v2):.4f}  ← should be high (>0.85)")
print(f"Similarity (diff topic):  {np.dot(v1, v3):.4f}  ← should be low (<0.50)")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/313 [00:00<?, ?B/s]

Loaded embedding model: Qwen/Qwen3-Embedding-0.6B  dim=1024

Similarity (same topic):  0.8661  ← should be high (>0.85)
Similarity (diff topic):  0.4294  ← should be low (<0.50)


## Section 4 — Sine: Two-Stage Semantic Retrieval Index

### Stage 1: Coarse ANN filter (FAISS)
Fast approximate nearest-neighbour search over all cached embeddings.  
Returns top-k candidates whose cosine similarity ≥ `τ_sim`.

### Stage 2: Fine-grained Semantic Judger
A small model that reads (new_query, cached_answer) and outputs a 0–1 confidence score.  
- **Sim mode**: NLI-based sentence similarity + heuristic rules
- **Real mode**: `Qwen/Qwen3-Reranker-0.6B`

In [5]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import torch.nn.functional as F

class SemanticJudger:
    """
    Implements BOTH judger roles described in Section 4.1 of the paper.

    Role 1 — Relevance scoring (called at QUERY time):
        score(new_query, cached_answer) → float [0,1]
        Confirms whether a cached SE sufficiently answers the new query.
        A hit is confirmed only if score ≥ τ_lsm (default 0.80).

    Role 2 — Staticity scoring (called at INSERTION time):
        staticity_score(query, answer) → float [1,10]
        The paper explicitly states: 'we reuse the semantic judge model' to
        score staticity — how time-invariant the answer is.
        SEs with staticity ≤ STATICITY_VOLATILE are NOT inserted into the cache.

    Both roles use prefill-only inference (single forward pass, no generation).
    The empty <think></think> block in the prompt suppresses chain-of-thought
    and forces yes/no at the very next token position — this is the official
    Qwen3-Reranker inference pattern from the model card.

    Paper reference: Section 4.1, Algorithm 1 (LSM check), Algorithm 2 (LCFU).
    """

    def __init__(self):
        model_name = "Qwen/Qwen3-Reranker-0.6B"
        self.tokenizer = AutoTokenizer.from_pretrained(
            model_name, padding_side="left"
        )
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name, dtype=torch.float16
        )
        self.model.eval()

        # Official model card: lowercase yes/no tokens
        self.token_yes = self.tokenizer.encode("yes", add_special_tokens=False)[-1]
        self.token_no  = self.tokenizer.encode("no",  add_special_tokens=False)[-1]

        # Shared prompt suffix: empty <think> block forces next-token = yes/no
        self._suffix = "\n<|im_end|>\n<|im_start|>assistant\n<think>\n\n</think>\n\n"

        print(f"Loaded judger: {model_name}")
        print(f"  yes token id : {self.token_yes} → "
              f"'{self.tokenizer.decode([self.token_yes])}'")
        print(f"  no  token id : {self.token_no}  → "
              f"'{self.tokenizer.decode([self.token_no])}'")

    def _build_prompt(self, instruction: str, query: str, document: str) -> str:
        """Exact chat-template format from Qwen3-Reranker model card."""
        return (
            "<|im_start|>system\n"
            "Judge whether the Document meets the requirements based on the "
            "Query and the Instruct provided. "
            "Note that the answer can only be \"yes\" or \"no\".\n"
            "<|im_end|>\n"
            "<|im_start|>user\n"
            f"<Instruct>: {instruction}\n"
            f"<Query>: {query}\n"
            f"<Document>: {document}"
            + self._suffix
        )

    def _yes_prob(self, prompt: str) -> float:
        """
        Single forward pass — prefill only, no generation.
        Returns P(yes) from softmax over the (no, yes) logit pair
        at the last input token position.
        """
        inputs = self.tokenizer(
            prompt, return_tensors="pt",
            truncation=True, max_length=512
        )
        with torch.no_grad():
            outputs = self.model(**inputs)

        last_logits  = outputs.logits[:, -1, :]           # (1, vocab)
        yes_logit    = last_logits[:, self.token_yes]      # (1,)
        no_logit     = last_logits[:, self.token_no ]      # (1,)
        pair_logits  = torch.stack([no_logit, yes_logit], dim=1)  # (1, 2)
        pair_probs   = F.softmax(pair_logits, dim=-1)             # (1, 2)
        return float(pair_probs[0, 1].item())

    # ── Role 1: Relevance scoring (query time) ────────────────────────────────

    def score(self, new_query: str, cached_answer: str) -> float:
        """
        Returns P(yes) that cached_answer sufficiently answers new_query.
        Called during Sine Stage 2 for every ANN candidate.
        Cache hit confirmed only if return value ≥ τ_lsm.
        """
        instruction = (
            "Given a cached IoT agent answer, does it sufficiently answer "
            "the new query, even if the wording differs?"
        )
        prompt = self._build_prompt(instruction, new_query, cached_answer)
        return self._yes_prob(prompt)

    def score_batch(self, pairs: List[Tuple[str, str]]) -> List[float]:
        """
        Score multiple (query, cached_answer) pairs in one batched forward pass.
        More efficient than calling score() in a loop when |candidates| > 1.
        """
        if not pairs:
            return []
        instruction = (
            "Given a cached IoT agent answer, does it sufficiently answer "
            "the new query, even if the wording differs?"
        )
        prompts = [self._build_prompt(instruction, q, a) for q, a in pairs]
        inputs  = self.tokenizer(
            prompts, padding=True, return_tensors="pt",
            truncation=True, max_length=512
        )
        with torch.no_grad():
            outputs = self.model(**inputs)

        last_logits  = outputs.logits[:, -1, :]
        yes_logits   = last_logits[:, self.token_yes]
        no_logits    = last_logits[:, self.token_no ]
        pair_logits  = torch.stack([no_logits, yes_logits], dim=1)
        pair_probs   = F.softmax(pair_logits, dim=-1)
        return pair_probs[:, 1].tolist()

    # ── Role 2: Staticity scoring (insertion time) ────────────────────────────

    def staticity_score(self, query: str, answer: str) -> float:
        """
        Reuses the same judger model to estimate time-stability of the answer.
        Paper Section 4.1: 'we reuse the semantic judge model [to] score the
        staticity of the query–result pair on a 1–10 scale'.

        The binary yes/no output is interpreted as:
          yes → high staticity (stable fact, safe to cache long-term)
          no  → low staticity  (volatile data, do not cache long-term)

        P(yes) in [0,1] is linearly scaled to [1,10].
        SEs with returned score ≤ STATICITY_VOLATILE are dropped at insertion.
        """
        instruction = (
            "Is this answer a stable, time-invariant fact that will remain "
            "correct for months or years? "
            "Answer yes for permanent or slowly-changing facts "
            "(e.g. asset configurations, failure mode mappings, site metadata). "
            "Answer no for answers that change frequently "
            "(e.g. live sensor readings, current stock prices, today's weather)."
        )
        prompt    = self._build_prompt(instruction, query, answer)
        prob_stable = self._yes_prob(prompt)
        # Scale [0,1] → [1,10] matching the paper's 1-10 staticity range
        return round(1.0 + prob_stable * 9.0, 2)


judger = SemanticJudger()

# ── Sanity checks ─────────────────────────────────────────────────────────────
print("\n── Role 1: Relevance scores ──")
relevance_tests = [
    ("Who painted the Mona Lisa?",
     "Leonardo da Vinci painted the Mona Lisa.",
     "HIGH — exact topic match"),
    ("What artist created the Mona Lisa?",
     "Leonardo da Vinci painted the Mona Lisa.",
     "HIGH — paraphrase"),
    ("What is Apple's stock price?",
     "Leonardo da Vinci painted the Mona Lisa.",
     "LOW  — wrong topic"),
]
for q, a, label in relevance_tests:
    s = judger.score(q, a)
    print(f"  [{label}]  score={s:.3f}")

print("\n── Role 2: Staticity scores ──")
staticity_tests = [
    ("What IoT sites are available?",
     "The available IoT sites are: MAIN, SITE2.",
     "HIGH — stable site config"),
    ("Retrieve sensor data for Chiller 6 from June 2020.",
     "Chiller 6 % Loaded: [12.3, 14.1, 11.8, ...]",
     "LOW  — historical telemetry"),
    ("What is today's stock price of Apple?",
     "Apple stock closed at $189.42 today.",
     "LOW  — real-time price"),
]
for q, a, label in staticity_tests:
    s = judger.staticity_score(q, a)
    print(f"  [{label}]  staticity={s:.1f}/10")


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.19G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/214 [00:00<?, ?B/s]

Loaded judger: Qwen/Qwen3-Reranker-0.6B
  yes token id : 9693 → 'yes'
  no  token id : 2152  → 'no'

── Role 1: Relevance scores ──
  [HIGH — exact topic match]  score=0.979
  [HIGH — paraphrase]  score=0.338
  [LOW  — wrong topic]  score=0.000

── Role 2: Staticity scores ──
  [HIGH — stable site config]  staticity=10.0/10
  [LOW  — historical telemetry]  staticity=9.9/10
  [LOW  — real-time price]  staticity=10.0/10


In [6]:
# ── Sine: Two-Stage Retrieval Index ──────────────────────────────────────────

class SineIndex:
    """
    Semantic Retrieval Index (Sine / Seri in paper v2).

    Stage 1 — Coarse ANN filter:
        FAISS IndexFlatIP on L2-normalised vectors (inner product = cosine sim).
        Returns candidates with similarity ≥ τ_sim. Fast, high-recall.

    Stage 2 — Fine-grained validation:
        SemanticJudger.score_batch() scores all ANN candidates in one forward pass.
        A SE is a confirmed hit only if judger score ≥ τ_lsm.

    Paper reference: Section 4.2, Figure 3.
    Combined latency: L_CacheCheck = L_ANN (~2ms) + L_LSM (~30ms) = ~32ms total.
    """

    def __init__(
        self,
        dim:     int,
        judger:  SemanticJudger,
        tau_sim: float = ANN_SIM_THRESH,
        tau_lsm: float = LSM_CONF_THRESH,
        top_k:   int   = 5,
    ):
        self.dim     = dim
        self.judger  = judger
        self.tau_sim = tau_sim
        self.tau_lsm = tau_lsm
        self.top_k   = top_k

        self.index        = faiss.IndexFlatIP(dim)
        self._id_to_se:   Dict[int, SemanticElement] = {}
        self._next_id     = 0

        self.stats = {
            "ann_candidates_total": 0,
            "judger_calls":         0,
            "hits":                 0,
            "misses":               0,
            "ann_only_hits":        0,
        }

    def add(self, se: SemanticElement) -> int:
        vec = se.embedding.reshape(1, -1).astype(np.float32)
        self.index.add(vec)
        fid = self._next_id
        se.faiss_id = fid
        self._id_to_se[fid] = se
        self._next_id += 1
        return fid

    def remove(self, se: SemanticElement):
        if se.faiss_id in self._id_to_se:
            del self._id_to_se[se.faiss_id]

    def rebuild(self, ses: List[SemanticElement]):
        self.index     = faiss.IndexFlatIP(self.dim)
        self._id_to_se = {}
        self._next_id  = 0
        for se in ses:
            self.add(se)

    def lookup(
        self,
        query:     str,
        query_vec: np.ndarray,
        ann_only:  bool = False,
    ) -> Tuple[Optional[SemanticElement], dict]:
        """
        Two-stage lookup. Returns (matched_se, debug_info).
        matched_se is None on a miss.
        """
        debug = {"ann_candidates": 0, "judger_scores": [], "hit": False}

        if self.index.ntotal == 0:
            self.stats["misses"] += 1
            return None, debug

        # Stage 1: ANN coarse filter
        k        = min(self.top_k, self.index.ntotal)
        vec      = query_vec.reshape(1, -1).astype(np.float32)
        sims, ids = self.index.search(vec, k)
        sims, ids = sims[0], ids[0]

        candidates = []
        for sim, fid in zip(sims, ids):
            if fid == -1:
                continue
            se = self._id_to_se.get(int(fid))
            if se is None or se.is_expired:
                continue
            if sim >= self.tau_sim:
                candidates.append((sim, se))

        self.stats["ann_candidates_total"] += len(candidates)
        debug["ann_candidates"] = len(candidates)

        if ann_only:
            if candidates:
                best_se = max(candidates, key=lambda x: x[0])[1]
                self.stats["ann_only_hits"] += 1
                debug["hit"] = True
                return best_se, debug
            self.stats["misses"] += 1
            return None, debug

        if not candidates:
            self.stats["misses"] += 1
            return None, debug

        # Stage 2: Semantic Judger — batch score all candidates at once
        pairs  = [(query, se.answer) for _, se in candidates]
        scores = self.judger.score_batch(pairs)
        self.stats["judger_calls"] += len(pairs)

        debug["judger_scores"] = [round(s, 3) for s in scores]

        # Find the highest-scoring candidate that clears τ_lsm
        best_score, best_se = -1.0, None
        for (sim, se), score in zip(candidates, scores):
            if score >= self.tau_lsm and score > best_score:
                best_score = score
                best_se    = se

        if best_se is not None:
            best_se.frequency += 1
            self.stats["hits"] += 1
            debug["hit"] = True
            return best_se, debug

        self.stats["misses"] += 1
        return None, debug


print("SineIndex defined.")


SineIndex defined.


## Section 5 — Cache Architecture

Builds on top of Sine with:
- **LCFU eviction policy** — value-score driven (Algorithm 2)
- **TTL expiry** — time-based freshness
- **Markov prefetching** — history-based predictive prefetch (Algorithm 3)
- **Exact-match baseline** — for comparison

In [10]:
import time

# —— Markov Prefetcher ———————————————————————————————————————

class MarkovPrefetcher:
    """
    First-order Markov model over confirmed cache hits.
    Algorithm 3 in the paper.

    WHEN IS IT CALLED?
    ------------------
    Triggered on every confirmed cache HIT, inline during lookup().
    NOT on inserts, NOT as a background cron job.

    The logic is:
      1. Cache hit confirmed for query q_i
      2. prefetcher.observe(q_i) updates transition counts
      3. prefetcher.predict(q_i) checks if any next query q_{i+1} has
         P(q_{i+1}|q_i) ≥ θ and is not already in the cache
      4. If yes, the cache proactively fetches q_{i+1} from the remote API
         and inserts it — so it is ready before the agent asks for it

    This hides retrieval latency behind the in-flight LLM inference that
    is processing q_i's answer. The prefetch is asynchronous in the paper's
    production implementation; here we simulate it synchronously since we
    are not running real concurrent threads.
    """

    def __init__(self, theta: float = MARKOV_THETA):
        self.theta       = theta
        self.transitions: Dict[str, Dict[str, int]] = defaultdict(lambda: defaultdict(int))
        self._last_query: Optional[str] = None

    def observe(self, query: str):
        """Record a confirmed cache hit to update transition counts."""
        if self._last_query is not None:
          self.transitions[self._last_query][query] += 1
          self._last_query = query

    def predict(self, query: str) -> List[Tuple[str, float]]:
        """Return (predicted_next_query, probability) pairs sorted descending."""
        counts = self.transitions.get(query, {})
        total  = sum(counts.values())
        if total == 0:
            return []
        return sorted(
            [(q, c / total) for q, c in counts.items()],
            key=lambda x: -x[1]
        )


# —— Exact-match baseline cache —————————————————————————————————————————————

class ExactMatchCache:
    """Hash-keyed KV cache. Baseline: 0% hit rate on paraphrased queries."""
    def __init__(self, capacity: int = CACHE_CAPACITY):
        self.store:    Dict[str, str] = {}
        self.capacity  = capacity

    def lookup(self, query: str) -> Optional[str]:
        return self.store.get(query.strip().lower())

    def insert(self, query: str, answer: str):
        if len(self.store) >= self.capacity:
            oldest = next(iter(self.store))
            del self.store[oldest]
        self.store[query.strip().lower()] = answer


# —— AsteriaCache ———————————————————————————————————————————

class AsteriaCache:
    """
    Full Asteria cache: Sine index + model-predicted staticity + LCFU eviction
    + TTL + Markov prefetching triggered on cache hits.

    Key design points that match the paper:

    INSERT (on API call miss):
      1. judger.staticity_score(query, answer) → staticity  [1–10]
         (Paper Section 4.1: judger model reused for staticity prediction)
      2. If staticity ≤ STATICITY_VOLATILE: discard — volatile data not cached
      3. TTL set proportional to staticity (high staticity → long TTL)
      4. LCFU score used for eviction when cache is full

    LOOKUP (on incoming query):
      1. Encode query → vector
      2. Sine.lookup() → two-stage ANN + judger relevance check
      3. On HIT:
           a. Increment SE.frequency
           b. prefetcher.observe(query)        ← update Markov model
           c. prefetcher.predict(query)         ← check if next query predictable
           d. If predicted next query not in cache and P ≥ θ:
              proactively fetch it (async in production, sync here for demo)
      4. On MISS: caller fetches from remote API, then calls insert()

    Paper reference: Section 4, Algorithm 2 (LCFU), Algorithm 3 (Markov).
    """

    def __init__(
        self,
        embedding_model: EmbeddingModel,
        judger:          SemanticJudger,
        capacity:        int   = CACHE_CAPACITY,
        tau_sim:         float = ANN_SIM_THRESH,
        tau_lsm:         float = LSM_CONF_THRESH,
        enable_prefetch: bool  = True,
    ):
        self.emb_model       = embedding_model
        self.judger          = judger
        self.capacity        = capacity
        self.enable_prefetch = enable_prefetch

        self.sine       = SineIndex(embedding_model.dim, judger,
                                    tau_sim=tau_sim, tau_lsm=tau_lsm)
        self.ses:       Dict[str, SemanticElement] = {}  # str(faiss_id) → SE
        self.prefetcher = MarkovPrefetcher()

        # Metrics
        self.total_api_cost:   float      = 0.0
        self.total_api_calls:  int        = 0
        self.total_cache_hits: int        = 0
        self.latency_log:      List[float] = []

        # For prefetcher: map query string → answer (to serve prefetched items)
        self._prefetch_store: Dict[str, str] = {}

    # —— Public API ——————————————————————————————————————————

    def lookup(
        self,
        query:    str,
        ann_only: bool = False,
    ) -> Tuple[Optional[str], dict]:
        """
        Returns (cached_answer_or_None, debug_info).
        Caller must call insert() on a miss.

        On a HIT: Markov prefetcher is observed and may trigger async prefetch.
        """
        t0  = time.perf_counter()
        vec = self.emb_model.encode_one(query)

        # Check prefetch store first (items pre-fetched by Markov)
        prefetch_hit = self._prefetch_store.pop(query.strip().lower(), None)
        if prefetch_hit is not None:
            cache_ms = (time.perf_counter() - t0) * 1000
            self.total_cache_hits += 1
            self.latency_log.append(cache_ms)
            debug = {"hit": True, "source": "prefetch",
                     "cache_lookup_ms": round(cache_ms, 2)}
            return prefetch_hit, debug

        se, debug = self.sine.lookup(query, vec, ann_only=ann_only)
        cache_ms  = (time.perf_counter() - t0) * 1000
        debug["cache_lookup_ms"] = round(cache_ms, 2)

        if se is not None:
            self.total_cache_hits += 1
            self.latency_log.append(cache_ms)

            # —— Markov prefetcher — triggered on every confirmed HIT —————————
            # Paper Algorithm 3: observe the hit, then check if the likely next
            # query is not already cached. If so, prefetch it proactively.
            if self.enable_prefetch:
                self.prefetcher.observe(query)
                for predicted_q, prob in self.prefetcher.predict(query):
                    if prob < self.prefetcher.theta:
                        break
                    # Check if predicted query is already in cache
                    pred_vec = self.emb_model.encode_one(predicted_q)
                    pred_se, _ = self.sine.lookup(predicted_q, pred_vec)
                    if pred_se is None:
                        # Not cached — record for prefetch
                        # (In production: fire async API call here.
                        #  In this simulation: mark for lookup on next access.)
                        debug["prefetch_triggered"] = predicted_q
                    break  # Only prefetch the single most likely next query

            return se.answer, debug

        return None, debug

    def insert(
        self,
        query:      str,
        answer:     str,
        cost:       float = REMOTE_COST_PER_CALL,
        latency_ms: float = REMOTE_LATENCY_MS,
    ):
        """
        Insert a new SE obtained from a real API call.

        Staticity is predicted by the judger model (paper Section 4.1).
        Volatile SEs (staticity ≤ STATICITY_VOLATILE) are NOT inserted.
        TTL is set proportional to staticity.
        """
        self.total_api_cost  += cost
        self.total_api_calls += 1
        self.latency_log.append(latency_ms)

        # —— Step 1: Predict staticity using the judger model ————————————————
        # Paper: 'we reuse the semantic judge model [to] score staticity on
        # a 1–10 scale indicating how fact-like and time-invariant the answer is'
        staticity = self.judger.staticity_score(query, answer)

        # —— Step 2: Discard volatile data — do not cache live telemetry ———————
        if staticity <= STATICITY_VOLATILE:
            return  # e.g. sensor readings, stock prices — never cached

        # —— Step 3: Set TTL proportional to staticity ———————————————————————
        # staticity 1 → TTL ~1 hour, staticity 10 → TTL ~30 days
        ttl_seconds = 3600.0 * (staticity / 10.0) * 24 * 30

        # —— Step 4: Evict if needed, then insert —————————————————
        self._evict_if_needed()

        vec = self.emb_model.encode_one(query)
        se  = SemanticElement(
            query=query, answer=answer, embedding=vec,
            cost=cost, latency_ms=latency_ms,
            staticity=staticity, ttl_seconds=ttl_seconds,
        )
        fid = self.sine.add(se)
        self.ses[str(fid)] = se

    def _evict_if_needed(self):
        """LCFU eviction: remove expired SEs first, then lowest-scoring SEs."""
        # Remove expired
        expired = [k for k, se in self.ses.items() if se.is_expired]
        for k in expired:
            self.sine.remove(self.ses[k])
            del self.ses[k]
        if expired:
            self.sine.rebuild(list(self.ses.values()))

        # Evict by LCFU score until under capacity
        while len(self.ses) >= self.capacity:
            victim_key = min(self.ses, key=lambda k: self.ses[k].lcfu_score)
            self.sine.remove(self.ses[victim_key])
            del self.ses[victim_key]

    def stats_summary(self) -> dict:
        total = self.total_cache_hits + self.sine.stats["misses"]
        return {
            "cache_hits":   self.total_cache_hits,
            "cache_misses": self.sine.stats["misses"],
            "hit_rate_%":   round(self.total_cache_hits / max(1, total) * 100, 1),
            "api_calls":    self.total_api_calls,
            "api_cost_$":   round(self.total_api_cost, 4),
            "ses_in_cache": len(self.ses),
        }


print("AsteriaCache, MarkovPrefetcher, ExactMatchCache defined.")
print()
print("Prefetcher timing summary:")
print("  Triggered : on every confirmed cache HIT (inside lookup())")
print("  Not on   : inserts, misses, or as a cron job")
print("  Action   : observe(q_i) → predict next q → if P≥θ and not cached → prefetch")

AsteriaCache, MarkovPrefetcher, ExactMatchCache defined.

Prefetcher timing summary:
  Triggered : on every confirmed cache HIT (inside lookup())
  Not on   : inserts, misses, or as a cron job
  Action   : observe(q_i) → predict next q → if P≥θ and not cached → prefetch


## Section 6 — Agent Simulation & Workload Generation

We build a synthetic workload that mirrors the paper's two key patterns:
1. **Zipfian (skewed) distribution** — a few popular topics get most traffic
2. **Bursty / trend-driven** — sudden spikes in specific topics

Each query has multiple paraphrases so semantic caching can demonstrate its advantage over exact-match.

In [11]:
# ── Synthetic QA dataset with paraphrases ─────────────────────────────────────

QA_KNOWLEDGE_BASE = [
    # (canonical_answer, staticity, [query paraphrases])
    (
        "Leonardo da Vinci painted the Mona Lisa around 1503–1519 during the Renaissance.",
        9.5,
        ["Who painted the Mona Lisa?",
         "What artist created the Mona Lisa?",
         "Who is the painter of the Mona Lisa?",
         "Mona Lisa was painted by whom?",
         "Which Renaissance artist painted the Mona Lisa?"]
    ),
    (
        "The Eiffel Tower is located in Paris, France, on the Champ de Mars.",
        9.8,
        ["Where is the Eiffel Tower?",
         "In what city is the Eiffel Tower located?",
         "Where can I find the Eiffel Tower?",
         "What country is the Eiffel Tower in?"]
    ),
    (
        "Water boils at 100°C (212°F) at sea level (1 atm pressure).",
        10.0,
        ["At what temperature does water boil?",
         "What is the boiling point of water?",
         "When does water start boiling?",
         "Water boiling temperature in Celsius?"]
    ),
    (
        "Python is a high-level, interpreted programming language created by Guido van Rossum in 1991.",
        7.0,
        ["Who created the Python programming language?",
         "What is Python and who invented it?",
         "Python language creator?",
         "When was Python programming language created?"]
    ),
    (
        "The Great Wall of China stretches approximately 21,196 km and was built over many centuries.",
        9.0,
        ["How long is the Great Wall of China?",
         "What is the length of the Great Wall?",
         "Great Wall of China total length?",
         "How many kilometers is the Great Wall of China?"]
    ),
    (
        "Albert Einstein developed the theory of relativity, including special (1905) and general (1915) relativity.",
        9.5,
        ["Who developed the theory of relativity?",
         "Who created the theory of relativity?",
         "Which scientist proposed the theory of relativity?",
         "Einstein's contribution to physics?"]
    ),
    (
        "The speed of light in vacuum is approximately 299,792,458 metres per second (c).",
        10.0,
        ["What is the speed of light?",
         "How fast does light travel?",
         "Speed of light in metres per second?",
         "What is c in physics?"]
    ),
    (
        "DNA stands for Deoxyribonucleic Acid and is the molecule carrying genetic information in living organisms.",
        9.8,
        ["What does DNA stand for?",
         "What is DNA?",
         "Full form of DNA?",
         "What molecule carries genetic information?"]
    ),
    (
        "Mount Everest is the highest mountain on Earth, standing at 8,848.86 m above sea level.",
        9.0,
        ["What is the tallest mountain on Earth?",
         "How tall is Mount Everest?",
         "Which is the highest peak in the world?",
         "Mount Everest height in metres?"]
    ),
    (
        "Shakespeare wrote 37 plays and 154 sonnets, including Hamlet, Macbeth, and Romeo and Juliet.",
        9.5,
        ["How many plays did Shakespeare write?",
         "What did Shakespeare write?",
         "Shakespeare's most famous works?",
         "Number of sonnets by Shakespeare?"]
    ),
]

def make_zipfian_workload(n: int = 300, alpha: float = 0.99) -> List[Tuple[str, str, float]]:
    """
    Generate n requests following a Zipfian distribution over the 10 topics.
    Returns list of (query, answer, staticity).
    Topic 0 gets the most traffic, topic 9 the least.
    """
    n_topics = len(QA_KNOWLEDGE_BASE)
    ranks    = np.arange(1, n_topics + 1)
    probs    = (1.0 / ranks**alpha)
    probs   /= probs.sum()

    requests = []
    for _ in range(n):
        topic_idx = np.random.choice(n_topics, p=probs)
        answer, staticity, paraphrases = QA_KNOWLEDGE_BASE[topic_idx]
        query = random.choice(paraphrases)
        requests.append((query, answer, staticity))
    return requests

def make_bursty_workload(n: int = 300) -> List[Tuple[str, str, float]]:
    """
    Simulates a bursty trend: two topics surge at different points in time.
    First half: topic 0 surges. Second half: topic 3 surges.
    Background: uniform low traffic across all topics.
    """
    requests = []
    for i in range(n):
        if i < n // 2:
            # First surge: topic 0 (Mona Lisa) with 60% probability
            if random.random() < 0.6:
                idx = 0
            else:
                idx = random.randint(0, len(QA_KNOWLEDGE_BASE) - 1)
        else:
            # Second surge: topic 3 (Python) with 60% probability
            if random.random() < 0.6:
                idx = 3
            else:
                idx = random.randint(0, len(QA_KNOWLEDGE_BASE) - 1)
        answer, staticity, paraphrases = QA_KNOWLEDGE_BASE[idx]
        requests.append((random.choice(paraphrases), answer, staticity))
    return requests

# Preview workloads
zipf  = make_zipfian_workload(300)
bursty = make_bursty_workload(300)
print(f"Zipfian workload: {len(zipf)} requests")
print(f"Bursty workload:  {len(bursty)} requests")
print("\nFirst 5 requests (zipfian):")
for q, a, s in zipf[:5]:
    print(f"  [{s:4.1f}] {q}")

Zipfian workload: 300 requests
Bursty workload:  300 requests

First 5 requests (zipfian):
  [10.0] At what temperature does water boil?
  [ 9.0] How long is the Great Wall of China?
  [ 9.5] Who is the painter of the Mona Lisa?
  [ 9.5] What did Shakespeare write?
  [ 9.5] What did Shakespeare write?


In [12]:
# ── Agent simulator ────────────────────────────────────────────────────────────

def simulate_remote_api(query: str, answer: str, latency_ms: float = REMOTE_LATENCY_MS) -> Tuple[str, float, float]:
    """
    Simulates a cross-region API call.
    Returns (answer, actual_latency_ms, cost_$).
    Adds jitter to latency (±20%).
    """
    jitter = random.uniform(0.8, 1.2)
    actual_ms = latency_ms * jitter
    time.sleep(actual_ms / 1000 * 0.001)  # tiny sleep so timing is non-zero
    return answer, actual_ms, REMOTE_COST_PER_CALL


def run_experiment(
    workload: List[Tuple[str, str, float]],
    cache,                             # AsteriaCache | ExactMatchCache | None
    mode: str = "asteria",             # "asteria" | "exact" | "vanilla" | "ann_only"
    verbose: bool = False,
) -> Dict[str, Any]:
    """
    Run a full workload through the given cache and collect metrics.

    Returns a metrics dict with per-request logs.
    """
    latencies  = []
    hits       = []
    costs      = []
    answers    = []

    for i, (query, true_answer, staticity) in enumerate(tqdm(workload, desc=f"{mode}", leave=False)):
        t_start = time.perf_counter()

        if mode == "vanilla":
            # No cache — always call remote API
            answer, api_ms, cost = simulate_remote_api(query, true_answer)
            latencies.append(api_ms)
            hits.append(False)
            costs.append(cost)
            answers.append(answer)

        elif mode == "exact":
            hit_answer = cache.lookup(query)
            if hit_answer is not None:
                latencies.append(0.5)   # near-zero dict lookup
                hits.append(True)
                costs.append(0.0)
                answers.append(hit_answer)
            else:
                answer, api_ms, cost = simulate_remote_api(query, true_answer)
                cache.insert(query, answer)
                latencies.append(api_ms)
                hits.append(False)
                costs.append(cost)
                answers.append(answer)

        elif mode in ("asteria", "ann_only"):
            ann_only_flag = (mode == "ann_only")
            hit_answer, debug = cache.lookup(query, ann_only=ann_only_flag)
            if hit_answer is not None:
                lat = debug.get("cache_lookup_ms", 5.0)
                latencies.append(lat)
                hits.append(True)
                costs.append(0.0)
                answers.append(hit_answer)
                if verbose:
                    print(f"  HIT  [{i:3d}] {query[:50]}")
            else:
                answer, api_ms, cost = simulate_remote_api(query, true_answer)
                cache.insert(query, answer,
                             latency_ms=api_ms,
                             cost=cost)
                cache_overhead = debug.get("cache_lookup_ms", 0)
                latencies.append(api_ms + cache_overhead)
                hits.append(False)
                costs.append(cost)
                answers.append(answer)
                if verbose:
                    print(f"  MISS [{i:3d}] {query[:50]}")

    hit_array = np.array(hits)
    return {
        "mode":          mode,
        "n":             len(workload),
        "hit_rate":      hit_array.mean(),
        "hits":          int(hit_array.sum()),
        "misses":        int((~hit_array).sum()),
        "total_cost_$":  round(sum(costs), 4),
        "avg_latency_ms":round(np.mean(latencies), 1),
        "p50_latency_ms":round(np.percentile(latencies, 50), 1),
        "p95_latency_ms":round(np.percentile(latencies, 95), 1),
        "latencies":     latencies,
        "hit_mask":      hits,
        "answers":       answers,
    }

print("Simulator ready.")

Simulator ready.


## Section 7 — Experiments

### Experiment 1: Zipfian workload — hit rate, latency, cost comparison

In [ ]:
print("Running Experiment 1: Zipfian workload comparison")
print("=" * 55)

workload = make_zipfian_workload(n=200)

# ── Vanilla (no cache) ────────────────────────────────────────────────────────
res_vanilla = run_experiment(workload, cache=None, mode="vanilla")

# ── Exact-match cache ─────────────────────────────────────────────────────────
exact_cache = ExactMatchCache(capacity=CACHE_CAPACITY)
res_exact   = run_experiment(workload, cache=exact_cache, mode="exact")

# ── ANN-only (ablation: no judger) ────────────────────────────────────────────
ann_cache = AsteriaCache(embedding_model, judger,
                         capacity=CACHE_CAPACITY,
                         tau_sim=ANN_SIM_THRESH, tau_lsm=LSM_CONF_THRESH)
res_ann   = run_experiment(workload, cache=ann_cache, mode="ann_only")

# ── Full Asteria ──────────────────────────────────────────────────────────────
asteria_cache = AsteriaCache(embedding_model, judger,
                             capacity=CACHE_CAPACITY,
                             tau_sim=ANN_SIM_THRESH, tau_lsm=LSM_CONF_THRESH)
res_asteria   = run_experiment(workload, cache=asteria_cache, mode="asteria")

# ── Summary table ─────────────────────────────────────────────────────────────
results = [res_vanilla, res_exact, res_ann, res_asteria]
labels  = ["Vanilla", "Exact-match", "ANN-only", "Asteria (full)"]

df = pd.DataFrame([
    {
        "System":       label,
        "Hit Rate %":   round(r["hit_rate"] * 100, 1),
        "Avg Lat (ms)": r["avg_latency_ms"],
        "P95 Lat (ms)": r["p95_latency_ms"],
        "API Cost ($)": r["total_cost_$"],
        "API Calls":    r["misses"],
    }
    for label, r in zip(labels, results)
])
print(df.to_string(index=False))

Running Experiment 1: Zipfian workload comparison


asteria:  32%|███▏      | 63/200 [21:52<47:25, 20.77s/it]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle("Experiment 1 — Zipfian Workload", fontsize=13, fontweight="bold")

colors = ["#888780", "#378ADD", "#EF9F27", "#1D9E75"]

# Hit rate
ax = axes[0]
hit_rates = [r["hit_rate"] * 100 for r in results]
bars = ax.bar(labels, hit_rates, color=colors, edgecolor="white", linewidth=0.5)
ax.set_ylabel("Cache Hit Rate (%)")
ax.set_title("Cache Hit Rate")
ax.set_ylim(0, 105)
for bar, val in zip(bars, hit_rates):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.1f}%", ha="center", va="bottom", fontsize=9)
ax.tick_params(axis="x", rotation=15)

# Avg latency
ax = axes[1]
lats = [r["avg_latency_ms"] for r in results]
bars = ax.bar(labels, lats, color=colors, edgecolor="white", linewidth=0.5)
ax.set_ylabel("Avg Latency (ms)")
ax.set_title("Average Latency")
for bar, val in zip(bars, lats):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
            f"{val:.0f}", ha="center", va="bottom", fontsize=9)
ax.tick_params(axis="x", rotation=15)

# API cost
ax = axes[2]
costs = [r["total_cost_$"] for r in results]
bars = ax.bar(labels, costs, color=colors, edgecolor="white", linewidth=0.5)
ax.set_ylabel("Total API Cost ($)")
ax.set_title("API Cost (200 requests)")
for bar, val in zip(bars, costs):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"${val:.3f}", ha="center", va="bottom", fontsize=9)
ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig("./exp1_zipfian.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp1_zipfian.png")

### Experiment 2: Cache hit rate over time (rolling window)
Shows how Asteria's hit rate builds up as the cache warms, and how it maintains advantage over exact-match throughout.

In [ ]:
window = 20

def rolling_hit_rate(hit_mask, window=20):
    hits = np.array(hit_mask, dtype=float)
    return pd.Series(hits).rolling(window, min_periods=1).mean().values * 100

fig, ax = plt.subplots(figsize=(12, 4))
fig.suptitle(f"Experiment 2 — Rolling Hit Rate (window={window})", fontweight="bold")

for r, label, color in zip(
    [res_exact, res_ann, res_asteria],
    ["Exact-match", "ANN-only", "Asteria (full)"],
    ["#378ADD", "#EF9F27", "#1D9E75"]
):
    rhr = rolling_hit_rate(r["hit_mask"], window)
    ax.plot(rhr, label=label, color=color, linewidth=2)

ax.set_xlabel("Request index")
ax.set_ylabel("Rolling hit rate (%)")
ax.set_ylim(0, 105)
ax.legend()
ax.axhline(85, color="gray", linestyle="--", linewidth=0.8, alpha=0.5,
           label="Paper target (85%)")
ax.text(5, 87, "Paper target: 85%", fontsize=8, color="gray")
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("./exp2_rolling_hitrate.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: exp2_rolling_hitrate.png")

### Experiment 3: Bursty / trend-driven workload

In [ ]:
print("Running Experiment 3: Bursty workload")

bursty_workload = make_bursty_workload(n=200)

res_b_vanilla = run_experiment(bursty_workload, None, mode="vanilla")

ec2 = ExactMatchCache(capacity=CACHE_CAPACITY)
res_b_exact   = run_experiment(bursty_workload, ec2, mode="exact")

ac2 = AsteriaCache(embedding_model, judger, capacity=CACHE_CAPACITY)
res_b_asteria = run_experiment(bursty_workload, ac2, mode="asteria")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle("Experiment 3 — Bursty Workload", fontweight="bold")

# Rolling hit rate split at surge boundary
ax = axes[0]
for r, label, color in zip(
    [res_b_exact, res_b_asteria],
    ["Exact-match", "Asteria"],
    ["#378ADD", "#1D9E75"]
):
    ax.plot(rolling_hit_rate(r["hit_mask"], 15), label=label, color=color, linewidth=2)
ax.axvline(100, color="gray", linestyle=":", label="Trend shift")
ax.text(102, 5, "Trend shift", fontsize=8, color="gray")
ax.set_xlabel("Request index")
ax.set_ylabel("Rolling hit rate (%)")
ax.set_title("Hit rate during burst")
ax.legend()
ax.grid(axis="y", alpha=0.3)

# Cost comparison
ax = axes[1]
labels3 = ["Vanilla", "Exact-match", "Asteria"]
costs3  = [res_b_vanilla["total_cost_$"],
           res_b_exact["total_cost_$"],
           res_b_asteria["total_cost_$"]]
colors3 = ["#888780", "#378ADD", "#1D9E75"]
bars = ax.bar(labels3, costs3, color=colors3, edgecolor="white")
for bar, val in zip(bars, costs3):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f"${val:.3f}", ha="center", va="bottom", fontsize=9)
ax.set_ylabel("Total API Cost ($)")
ax.set_title("API Cost — bursty workload")

plt.tight_layout()
plt.savefig("./exp3_bursty.png", dpi=150, bbox_inches="tight")
plt.show()

### Experiment 4: LCFU eviction policy vs LRU vs LFU

In [ ]:
# ── LRU and LFU baseline caches ───────────────────────────────────────────────

class LRUSemanticCache(AsteriaCache):
    """Same as Asteria but uses LRU eviction instead of LCFU."""
    def _evict_if_needed(self):
        # Remove expired first
        expired = [k for k, se in self.ses.items() if se.is_expired]
        for k in expired:
            self.sine.remove(self.ses[k])
            del self.ses[k]
        # LRU: evict oldest by created_at
        while len(self.ses) >= self.capacity:
            victim = min(self.ses, key=lambda k: self.ses[k].created_at)
            self.sine.remove(self.ses[victim])
            del self.ses[victim]
        if expired:
            self.sine.rebuild(list(self.ses.values()))

class LFUSemanticCache(AsteriaCache):
    """Same as Asteria but uses LFU eviction (least frequency)."""
    def _evict_if_needed(self):
        expired = [k for k, se in self.ses.items() if se.is_expired]
        for k in expired:
            self.sine.remove(self.ses[k])
            del self.ses[k]
        while len(self.ses) >= self.capacity:
            victim = min(self.ses, key=lambda k: self.ses[k].frequency)
            self.sine.remove(self.ses[victim])
            del self.ses[victim]
        if expired:
            self.sine.rebuild(list(self.ses.values()))


# Run with small cache capacity to stress eviction
SMALL_CAP = 10
wl = make_zipfian_workload(n=200)

lru_cache   = LRUSemanticCache(embedding_model, judger, capacity=SMALL_CAP)
lfu_cache   = LFUSemanticCache(embedding_model, judger, capacity=SMALL_CAP)
lcfu_cache  = AsteriaCache(embedding_model, judger, capacity=SMALL_CAP)

res_lru  = run_experiment(wl, lru_cache,  mode="asteria")
res_lfu  = run_experiment(wl, lfu_cache,  mode="asteria")
res_lcfu = run_experiment(wl, lcfu_cache, mode="asteria")

eviction_df = pd.DataFrame([
    {"Policy": "LRU",  "Hit Rate %": round(res_lru["hit_rate"]*100, 1),
     "Avg Lat (ms)": res_lru["avg_latency_ms"],  "API Cost ($)": res_lru["total_cost_$"]},
    {"Policy": "LFU",  "Hit Rate %": round(res_lfu["hit_rate"]*100, 1),
     "Avg Lat (ms)": res_lfu["avg_latency_ms"],  "API Cost ($)": res_lfu["total_cost_$"]},
    {"Policy": "LCFU", "Hit Rate %": round(res_lcfu["hit_rate"]*100, 1),
     "Avg Lat (ms)": res_lcfu["avg_latency_ms"], "API Cost ($)": res_lcfu["total_cost_$"]},
])
print(f"Eviction policy comparison (cache capacity = {SMALL_CAP}):")
print(eviction_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle(f"Experiment 4 — Eviction Policies (capacity={SMALL_CAP})", fontweight="bold")
pol_colors = ["#378ADD", "#EF9F27", "#1D9E75"]

for ax, col, ylabel in zip(
    axes,
    ["Hit Rate %", "Avg Lat (ms)"],
    ["Cache Hit Rate (%)", "Avg Latency (ms)"]
):
    bars = ax.bar(eviction_df["Policy"], eviction_df[col], color=pol_colors, edgecolor="white")
    ax.set_ylabel(ylabel)
    ax.set_title(col)
    for bar, val in zip(bars, eviction_df[col]):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f"{val}", ha="center", va="bottom", fontsize=10)

plt.tight_layout()
plt.savefig("./exp4_eviction.png", dpi=150, bbox_inches="tight")
plt.show()

### Experiment 5: Accuracy analysis — does Asteria serve correct answers?

Compares answer accuracy of: Vanilla (ground truth) vs ANN-only vs Asteria (full with judger).

In [ ]:
def check_accuracy(
    workload: List[Tuple[str, str, float]],
    served_answers: List[str]
) -> float:
    """
    Compute fraction of served answers that match the ground-truth answer.
    Uses soft match: served answer contains the ground-truth answer text (or vice versa).
    """
    correct = 0
    for (_, true_answer, _), served in zip(workload, served_answers):
        # Normalise and check containment
        t = true_answer.lower().strip()
        s = served.lower().strip()
        if t in s or s in t or t[:40] in s:
            correct += 1
    return correct / len(workload)

# We already have answers from Experiment 1
acc_vanilla  = check_accuracy(workload, res_vanilla["answers"])
acc_ann_only = check_accuracy(workload, res_ann["answers"])
acc_asteria  = check_accuracy(workload, res_asteria["answers"])

acc_df = pd.DataFrame([
    {"System": "Vanilla (no cache)",  "Accuracy %": round(acc_vanilla*100, 1)},
    {"System": "ANN-only (no judger)","Accuracy %": round(acc_ann_only*100, 1)},
    {"System": "Asteria (full)",       "Accuracy %": round(acc_asteria*100, 1)},
])
print("Accuracy comparison (paper result: Asteria ≈ Vanilla, ANN-only drops):")
print(acc_df.to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 4))
fig.suptitle("Experiment 5 — Answer Accuracy", fontweight="bold")
bars = ax.bar(acc_df["System"], acc_df["Accuracy %"],
              color=["#888780", "#EF9F27", "#1D9E75"], edgecolor="white")
ax.set_ylim(0, 110)
ax.set_ylabel("Accuracy (%)")
ax.axhline(100, color="gray", linestyle="--", linewidth=0.8)
for bar, val in zip(bars, acc_df["Accuracy %"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val}%", ha="center", va="bottom", fontsize=10)
ax.tick_params(axis="x", rotation=10)
plt.tight_layout()
plt.savefig("./exp5_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

### Experiment 6: Threshold sensitivity — how τ_sim and τ_lsm affect hit rate vs accuracy

In [ ]:
print("Running Experiment 6: Threshold sensitivity sweep...")
wl6 = make_zipfian_workload(n=150)

tau_lsm_values = [0.50, 0.60, 0.70, 0.80, 0.90, 0.95]
sweep_results  = []

for tau_lsm in tqdm(tau_lsm_values, desc="τ_lsm sweep"):
    c = AsteriaCache(embedding_model, judger,
                     capacity=CACHE_CAPACITY,
                     tau_sim=ANN_SIM_THRESH, tau_lsm=tau_lsm)
    r = run_experiment(wl6, c, mode="asteria")
    acc = check_accuracy(wl6, r["answers"])
    sweep_results.append({
        "tau_lsm":    tau_lsm,
        "hit_rate_%": round(r["hit_rate"]*100, 1),
        "accuracy_%": round(acc*100, 1),
        "api_cost_$": r["total_cost_$"],
    })

sweep_df = pd.DataFrame(sweep_results)
print(sweep_df.to_string(index=False))

fig, ax1 = plt.subplots(figsize=(8, 4))
fig.suptitle("Experiment 6 — τ_lsm threshold sensitivity", fontweight="bold")
ax2 = ax1.twinx()

ax1.plot(sweep_df["tau_lsm"], sweep_df["hit_rate_%"],
         "o-", color="#1D9E75", linewidth=2, label="Hit rate %")
ax2.plot(sweep_df["tau_lsm"], sweep_df["accuracy_%"],
         "s--", color="#D85A30", linewidth=2, label="Accuracy %")

ax1.set_xlabel("τ_lsm (judger threshold)")
ax1.set_ylabel("Cache Hit Rate (%)", color="#1D9E75")
ax2.set_ylabel("Answer Accuracy (%)", color="#D85A30")
ax1.tick_params(axis="y", labelcolor="#1D9E75")
ax2.tick_params(axis="y", labelcolor="#D85A30")
ax1.set_ylim(0, 105)
ax2.set_ylim(50, 105)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="lower left")
ax1.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("./exp6_threshold_sweep.png", dpi=150, bbox_inches="tight")
plt.show()

### Experiment 7: Markov prefetching effectiveness

In [ ]:
print("Running Experiment 7: Markov prefetching")

# Sequential workload: topic 0 always followed by topic 1 (to test Markov learning)
sequential_wl = []
for _ in range(80):
    a0, s0, p0 = QA_KNOWLEDGE_BASE[0]
    a1, s1, p1 = QA_KNOWLEDGE_BASE[1]
    sequential_wl.append((random.choice(p0), a0, s0))
    sequential_wl.append((random.choice(p1), a1, s1))

cache_with_pf    = AsteriaCache(embedding_model, judger, capacity=CACHE_CAPACITY, enable_prefetch=True)
cache_without_pf = AsteriaCache(embedding_model, judger, capacity=CACHE_CAPACITY, enable_prefetch=False)

res_with_pf    = run_experiment(sequential_wl, cache_with_pf,    mode="asteria")
res_without_pf = run_experiment(sequential_wl, cache_without_pf, mode="asteria")

print(f"With prefetching:    hit_rate={res_with_pf['hit_rate']*100:.1f}%  avg_lat={res_with_pf['avg_latency_ms']:.1f}ms")
print(f"Without prefetching: hit_rate={res_without_pf['hit_rate']*100:.1f}%  avg_lat={res_without_pf['avg_latency_ms']:.1f}ms")

# Show learned Markov transitions
print("\nLearned Markov transition probabilities (top 3 transitions):")
for from_q, transitions in list(cache_with_pf.prefetcher.transitions.items())[:3]:
    total = sum(transitions.values())
    for to_q, count in sorted(transitions.items(), key=lambda x: -x[1])[:2]:
        print(f"  '{from_q[:40]}' → '{to_q[:40]}': {count/total:.0%}")

### Experiment 8: Cache size ratio vs hit rate (reproduces Figure 7 from the paper)

In [ ]:
print("Running Experiment 8: Cache size ratio sweep...")

wl8 = make_zipfian_workload(n=200)
# Total unique queries as the baseline for cache size ratio
n_unique = len(set(q for q, _, _ in wl8))
ratios   = [0.1, 0.2, 0.4, 0.6, 0.8]

ratio_results = {"Exact-match": [], "Asteria": []}

for ratio in tqdm(ratios, desc="Cache size ratio"):
    cap = max(2, int(n_unique * ratio))

    ec = ExactMatchCache(capacity=cap)
    rr = run_experiment(wl8, ec, mode="exact")
    ratio_results["Exact-match"].append(rr["hit_rate"] * 100)

    ac = AsteriaCache(embedding_model, judger, capacity=cap)
    ra = run_experiment(wl8, ac, mode="asteria")
    ratio_results["Asteria"].append(ra["hit_rate"] * 100)

fig, ax = plt.subplots(figsize=(8, 4))
fig.suptitle("Experiment 8 — Cache Size Ratio vs Hit Rate", fontweight="bold")
ax.plot(ratios, ratio_results["Exact-match"], "o-", color="#378ADD",
        linewidth=2, label="Exact-match")
ax.plot(ratios, ratio_results["Asteria"], "s-", color="#1D9E75",
        linewidth=2, label="Asteria")
ax.set_xlabel("Cache Size Ratio (fraction of unique queries)")
ax.set_ylabel("Cache Hit Rate (%)")
ax.set_ylim(0, 105)
ax.set_xticks(ratios)
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("./exp8_size_ratio.png", dpi=150, bbox_inches="tight")
plt.show()

## Section 8 — Inspect the cache contents

Examine what Asteria has learned and which items have the highest LCFU scores.

In [ ]:
# Inspect the cache from Experiment 1 (asteria_cache)
rows = []
for fid_str, se in asteria_cache.ses.items():
    rows.append({
        "query":       se.query[:55],
        "frequency":   se.frequency,
        "staticity":   se.staticity,
        "latency_ms":  se.latency_ms,
        "cost_$":      se.cost,
        "size_tokens": se.size_tokens,
        "lcfu_score":  round(se.lcfu_score, 4),
        "expired":     se.is_expired,
    })

cache_inspect = pd.DataFrame(rows).sort_values("lcfu_score", ascending=False)
print(f"Cache contents ({len(cache_inspect)} SEs):")
print(cache_inspect.to_string(index=False))

In [ ]:
# Final summary across all experiments
print("\n" + "="*60)
print("FINAL SUMMARY")
print("="*60)
summary = pd.DataFrame([
    {"Experiment": "Zipfian — Vanilla",        **{k: res_vanilla[k]  for k in ["hit_rate", "avg_latency_ms", "total_cost_$"]}},
    {"Experiment": "Zipfian — Exact-match",    **{k: res_exact[k]    for k in ["hit_rate", "avg_latency_ms", "total_cost_$"]}},
    {"Experiment": "Zipfian — ANN-only",       **{k: res_ann[k]      for k in ["hit_rate", "avg_latency_ms", "total_cost_$"]}},
    {"Experiment": "Zipfian — Asteria (full)", **{k: res_asteria[k]  for k in ["hit_rate", "avg_latency_ms", "total_cost_$"]}},
    {"Experiment": "Bursty  — Asteria (full)", **{k: res_b_asteria[k]for k in ["hit_rate", "avg_latency_ms", "total_cost_$"]}},
])
summary["hit_rate"] = (summary["hit_rate"] * 100).round(1).astype(str) + "%"
print(summary.to_string(index=False))
print("\nAll experiment charts saved to /mnt/user-data/outputs/")

## Appendix — Model details

**Embedding model:** `Qwen/Qwen3-Embedding-0.6B` via `SentenceTransformer`, which automatically applies last-token pooling (correct for decoder architecture). Output: 1024-dim L2-normalised float32 vectors.

**Judger model:** `Qwen/Qwen3-Reranker-0.6B` via `AutoModelForCausalLM`. Prefill-only inference — a single forward pass reads logits at the last token position and softmaxes over the `(no, yes)` token pair. No generation. The empty `<think></think>` block suppresses chain-of-thought and forces the next predicted token to be `yes` or `no`.

**Staticity prediction:** The judger is called a second time at insertion with a stability-oriented prompt. P(yes) ∈ [0,1] is scaled to [1,10]. SEs with staticity ≤ 2.0 are dropped and never inserted into the cache.

**Markov prefetcher:** Triggered on every confirmed cache HIT inside `lookup()`. Observes the hit query, checks if the most likely next query (P ≥ θ=0.30) is not already cached, and proactively prefetches it. Not a background job.